# Banknote Authentication — kNN Modeling

This notebook prepares the dataset for machine learning, implements a k-Nearest Neighbors classifier from scratch, selects the hyperparameter `k`, and evaluates the final model.

The modeling workflow consists of:
- separating features and target,
- splitting the data into training, validation, and test sets,
- standardizing numerical features,
- implementing kNN from scratch,
- selecting the best value of `k`,
- evaluating the final classifier.

In [1]:
from pathlib import Path
import random

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = project_root / "data" / "raw" / "data_banknote_authentication.txt"
print(data_path)


/Users/oliversulej/Desktop/github_projects/banknote-authentication-from-scratch/data/raw/data_banknote_authentication.txt


## Preparing Features and Target

Before building the classifier, we separate the input features from the target variable.

The feature matrix `X` contains the four numerical attributes used by the model, while the target vector `y` contains the class labels that we want to predict.

In [3]:
column_names = [
    "variance",
    "skewness",
    "curtosis",
    "entropy",
    "class"
]

df = pd.read_csv(data_path, names=column_names)

In [4]:
features_names = ["variance", "skewness", "curtosis", "entropy"]

X = df[features_names]
y = df["class"]

## Train, Validation, and Test Split

Before training the classifier, the dataset must be divided into three independent subsets:

- **Training set** — used by the model to learn from known observations.
- **Validation set** — used to compare different values of the hyperparameter `k` and select the best configuration.
- **Test set** — used only once at the end to estimate how well the final model generalizes to previously unseen data.

To preserve the original class proportions in each subset, we will perform a **stratified split**, meaning that observations from class 0 and class 1 will be divided separately and then combined.

### Separating Class Indices

To create a stratified split manually, we first collect the row indices belonging to each target class.  
These indices will later be shuffled and divided into training, validation, and test subsets.

In [6]:
class_0_indices = df[df["class"]==0].index.to_list()
class_1_indices = df[df["class"]==1].index.to_list()

print(len(class_0_indices))
print(len(class_1_indices))

762
610


### Shuffling Class Indices

Before splitting the observations into training, validation, and test sets, the indices of each class are shuffled independently.

A fixed random seed is used to make the split reproducible, meaning that the same observations will be assigned to the same subsets every time the notebook is run.

In [7]:
rng = random.Random(42)

rng.shuffle(class_0_indices)
rng.shuffle(class_1_indices)



### Splitting Each Class

Each target class is divided separately into training, validation, and test subsets.

We use approximately:
- **70%** of observations for training,
- **15%** for validation,
- **15%** for final testing.

Splitting each class separately preserves a similar class distribution in all three subsets.

In [8]:
class_0_train_size = int(0.7*len(class_0_indices))
class_0_validation_size = int(0.15*len(class_0_indices))

class_1_train_size = int(0.7*len(class_1_indices))
class_1_validation_size = int(0.15*len(class_1_indices))

print(f"Class 0 training set size: {class_0_train_size}")
print(f"Class 0 validation set size: {class_0_validation_size}")
print(f"Class 1 training set size: {class_1_train_size}")
print(f"Class 1 validation set size: {class_1_validation_size}")

Class 0 training set size: 533
Class 0 validation set size: 114
Class 1 training set size: 427
Class 1 validation set size: 91


### Creating Train, Validation, and Test Indices

Using the shuffled class-specific indices, we now create separate training, validation, and test index lists.

The split is performed independently for each class and the resulting parts are then combined. This keeps the class proportions approximately consistent across all three subsets.

In [13]:
class_0_train_indices = class_0_indices[:class_0_train_size]
class_0_validation_indices = class_0_indices[class_0_train_size:class_0_train_size+class_0_validation_size]
class_0_test_indices = class_0_indices[class_0_train_size+class_0_validation_size:]

class_1_train_indices = class_1_indices[:class_1_train_size]
class_1_validation_indices = class_1_indices[class_1_train_size:class_1_train_size+class_1_validation_size]
class_1_test_indices = class_1_indices[class_1_train_size+class_1_validation_size:]

train_indices = class_0_train_indices + class_1_train_indices
validation_indices = class_0_validation_indices + class_1_validation_indices
test_indices = class_0_test_indices + class_1_test_indices

rng.shuffle(train_indices)
rng.shuffle(validation_indices)
rng.shuffle(test_indices)

print(len(train_indices))
print(len(validation_indices))
print(len(test_indices))

960
205
207
